In [7]:
import os
import decord
from tqdm import tqdm

import pandas as pd

In [8]:
path = '/mnt/sda1/saksham/TI2AV/others/AVSync15/metadata.csv'
df = pd.read_csv(path)

In [9]:
def is_video_valid(path):
    """
    Returns the number of frames, FPS, and duration (seconds) of a video.
    """    
    video_reader = decord.VideoReader(uri=path)
    video_num_frames = len(video_reader)
    fps = video_reader.get_avg_fps()  # Get frames per second (FPS)
    video_duration = video_num_frames / fps if fps > 0 else 0
    if video_num_frames>=121:
        return True
    else:
        return False 

base_dir = "/mnt/sda1/saksham/TI2AV/others/AVSync15/videos"
tqdm.pandas()
df['is_valid'] = df.progress_apply(lambda x: is_video_valid(os.path.join(base_dir, x['label'], x['vid']+'.mp4')), axis=1)
# df['is_valid'] = df['vid'].apply(lambda x: is_video_valid(os.path.join(base_dir, x['label'], x['vid']+'.mp4')), axis=1)

100%|██████████| 1500/1500 [00:28<00:00, 52.23it/s]


In [10]:
df_valid = df[df['is_valid'] == True].reset_index(drop=True)
del df_valid['is_valid']

df_valid_train = df_valid[df_valid['split'] == 'train'].reset_index(drop=True)
df_valid_test = df_valid[df_valid['split'] == 'test'].reset_index(drop=True)

In [11]:
len(df_valid), len(df_valid_train), len(df_valid_test)

(446, 397, 49)

In [12]:
save_path = '/home/sxk230060/TI2AV/misc/finetrainers/asva_scripts/AVSync15_metadata_valid.csv'
df.to_csv(save_path, index=False)